In [ ]:
# from groq import Groq
from dotenv import load_dotenv
from openai import OpenAI
import os
import json

load_dotenv()

True

In [48]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [51]:
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


In [52]:
def build_few_shot_prompt(venue, event):
    return f"""
You are a venue capacity estimation expert focused on France.

Below are examples of venues in France:

Example 1:
Venue: Parc des Princes, Paris
Event: Football Match
Output:
{{
  "venue_type": "stadium",
  "capacity_range": [40000, 50000],
  "estimated_capacity": 48000,
  "confidence": "high"
}}

Example 2:
Venue: Accor Arena, Paris
Event: Music Concert
Output:
{{
  "venue_type": "indoor arena",
  "capacity_range": [15000, 20000],
  "estimated_capacity": 18000,
  "confidence": "high"
}}

Example 3:
Venue: Palais des Congrès de Paris
Event: Business Conference
Output:
{{
  "venue_type": "conference center",
  "capacity_range": [2000, 5000],
  "estimated_capacity": 3500,
  "confidence": "medium"
}}

Example 4:
Venue: Zénith de Lyon
Event: Live Concert
Output:
{{
  "venue_type": "concert hall",
  "capacity_range": [3000, 7000],
  "estimated_capacity": 5000,
  "confidence": "high"
}}

Example 5:
Venue: Carrousel du Louvre, Paris
Event: Art Exhibition
Output:
{{
  "venue_type": "exhibition space",
  "capacity_range": [1000, 3000],
  "estimated_capacity": 2000,
  "confidence": "medium"
}}

Now estimate for the following:

Venue: {venue}
Event: {event}

Return ONLY JSON:
{{
  "venue_type": "",
  "capacity_range": [min, max],
  "estimated_capacity": number,
  "confidence": "low/medium/high"
}}
"""

In [55]:
def estimate_capacity(venue, event):
    response = client.responses.create(
        model="llama-3.1-8b-instant",
        temperature=0.2,
        input=[
            {
                "role": "system",
                "content": "You are a precise data estimation expert. Always return valid JSON."
            },
            {
                "role": "user",
                "content": build_few_shot_prompt(venue, event)
            }
        ]
    )

    text = response.output[1].content[0].text

    try:
        return json.loads(text)
    except:
        return {"raw": text}


In [58]:
print(estimate_capacity("KIOSQUE JARDIN DU RANELAGH", "CONCERT KIOSQUE RANELAGH"))
print(estimate_capacity("Gaddafi stadium lahore", "Pakistan Super League Final Match"))

{'venue_type': 'concert hall', 'capacity_range': [500, 1500], 'estimated_capacity': 1200, 'confidence': 'medium'}
{'venue_type': 'stadium', 'capacity_range': [30000, 40000], 'estimated_capacity': 35000, 'confidence': 'medium'}
